# Graph-Based Recommender — Personalized PageRank on the Bipartite User–Movie Graph

We build a weighted bipartite graph in which user and movie nodes are connected by an edge whose weight is the user's rating, and run **Personalized PageRank (PPR)** seeded on each test user. Items with the highest PPR mass under that personalization are the recommended items.

$$ \mathbf{p}^{(t+1)} = \alpha \, \mathbf{M}^\top \mathbf{p}^{(t)} + (1 - \alpha) \, \mathbf{v} $$

where `M` is the row-stochastic transition matrix derived from the rating-weighted bipartite adjacency, `v` is a one-hot vector at the target user node, and `α` is the damping factor (we use `0.85`, the standard PageRank value).

**Why this replaces the previous notebook.** The original `graph.ipynb` contained only matplotlib code plotting hard-coded RMSE / MAE / Precision / Recall / F1 / NDCG values comparing the other six methods — there was **no graph-based recommender** in this repository. We add one here so the project actually has the graph-based baseline its report claims.

**Rating prediction calibration.** Graph methods produce ranking signals on `[0, 1]`, not ratings on `[0.5, 5.0]`. We map PPR scores to the rating scale per user via Z-score scaling around the user's training mean, capped to the rating range. This is documented as a known limitation; the ranking metrics (HR@10, NDCG@10) are the primary evaluation for this baseline.

## What this notebook produces

- Predictions on the test set with `predicted_rating` per (user, movie), saved to `predictions/graph.csv`.
- Ranking metrics (HR@10, NDCG@10, etc.) via the shared `recsys_metrics` module.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
import os, random, sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import coo_matrix, csr_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR    = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR      = ARTIFACT_DIR / 'splits'
PREDICTIONS_DIR = ARTIFACT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(ARTIFACT_DIR))
from recsys_metrics import build_candidate_lists, evaluate_model

In [3]:
train = pd.read_csv(SPLITS_DIR / 'train.csv')
val   = pd.read_csv(SPLITS_DIR / 'val.csv')
test  = pd.read_csv(SPLITS_DIR / 'test.csv')
trainval = pd.concat([train, val], ignore_index=True)
print(f'train: {len(train):,}  val: {len(val):,}  test: {len(test):,}')

train: 99,616  val: 610  test: 610


## Build bipartite adjacency matrix

Indexing convention: nodes `0 .. n_users-1` are users, `n_users .. n_users + n_movies - 1` are movies.

In [4]:
users  = sorted(trainval['userId'].unique())
movies = sorted(trainval['movieId'].unique())
u2i = {u: i for i, u in enumerate(users)}
m2i = {m: i for i, m in enumerate(movies)}
n_users, n_movies = len(users), len(movies)
N = n_users + n_movies

# Edges: (user_node, movie_node, weight=rating) and the symmetric reverse direction.
u_idx = trainval['userId'].map(u2i).to_numpy()
m_idx = trainval['movieId'].map(m2i).to_numpy() + n_users
w     = trainval['rating'].to_numpy(dtype=np.float32)

rows = np.concatenate([u_idx, m_idx])
cols = np.concatenate([m_idx, u_idx])
data = np.concatenate([w, w])
A = coo_matrix((data, (rows, cols)), shape=(N, N)).tocsr()

# Row-normalize to get the transition matrix.
row_sums = np.asarray(A.sum(axis=1)).flatten()
row_sums_safe = np.where(row_sums > 0, row_sums, 1.0)
D_inv = csr_matrix((1.0 / row_sums_safe, (np.arange(N), np.arange(N))), shape=(N, N))
M = (D_inv @ A).tocsr()  # row-stochastic transition matrix
M_T = M.T.tocsr()         # PPR uses M^T for the right-multiplication form

print(f'Graph: {N} nodes ({n_users} users + {n_movies} movies), {A.nnz:,} directed edges')

Graph: 10311 nodes (610 users + 9701 movies), 200,452 directed edges


## Personalized PageRank (power iteration)

For each query user we precompute their PPR vector once and read off scores for any candidate movie node.

In [5]:
def ppr(user_node, alpha=0.85, max_iter=50, tol=1e-6):
    v = np.zeros(N, dtype=np.float32)
    v[user_node] = 1.0
    p = v.copy()
    for _ in range(max_iter):
        p_next = alpha * (M_T @ p) + (1 - alpha) * v
        if np.abs(p_next - p).sum() < tol:
            p = p_next
            break
        p = p_next
    return p

# Cache PPR vectors per (test) user.
ppr_cache = {}
test_users = test['userId'].unique()
for u in test_users:
    if u in u2i:
        ppr_cache[u] = ppr(u2i[u])
print(f'Cached PPR for {len(ppr_cache)} users.')

Cached PPR for 610 users.


## Score → rating mapping

For a candidate movie `j` and target user `u`:
1. Score = PPR score at `M[j]` minus PPR score the user gives to themselves (self-mass).
2. Z-score this score using the user's own PPR distribution over unrated movies.
3. Map to a rating: `μ_u + 0.5 * tanh(z)` clamped to `[0.5, 5.0]` (where `μ_u` is the user's training mean rating).

The `0.5` scale was chosen as a reasonable default (typical user rating std on MovieLens is ~1.0; we deliberately attenuate to avoid the graph signal overpowering the user-mean prior).

In [6]:
user_mean_rating = trainval.groupby('userId')['rating'].mean().to_dict()
global_mean = float(trainval['rating'].mean())

def _ppr_score(user_id, movie_id):
    if user_id not in ppr_cache or movie_id not in m2i:
        return None
    return float(ppr_cache[user_id][m2i[movie_id] + n_users])

def predict_rating(user_id, movie_id):
    s = _ppr_score(user_id, movie_id)
    if s is None:
        return global_mean
    # Z-score against the user's full distribution over all movie nodes.
    movie_block = ppr_cache[user_id][n_users:]
    mu, sd = movie_block.mean(), movie_block.std()
    z = (s - mu) / (sd + 1e-9)
    base = user_mean_rating.get(user_id, global_mean)
    return float(np.clip(base + 0.5 * np.tanh(z), 0.5, 5.0))

def predict_pointwise(df):
    out = df[['userId', 'movieId', 'rating']].copy()
    out['predicted_rating'] = [predict_rating(u, m) for u, m in zip(df['userId'], df['movieId'])]
    return out

def predict_fn(user_id, movie_ids):
    # For ranking, the raw PPR score on the movie node is the correct ranking signal.
    if user_id not in ppr_cache:
        return np.zeros(len(movie_ids))
    pvec = ppr_cache[user_id]
    out = np.zeros(len(movie_ids), dtype=np.float32)
    for k, m in enumerate(movie_ids):
        if int(m) in m2i:
            out[k] = pvec[m2i[int(m)] + n_users]
    return out

## Evaluate + save

In [7]:
candidates = build_candidate_lists(train, test, num_negatives=99, seed=SEED)
pointwise = predict_pointwise(test)
metrics = evaluate_model(predict_fn, test, candidates, pointwise_predictions=pointwise, k=10, threshold=3.5)
print('=== Personalized PageRank — test set ===')
for key in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']:
    print(f'  {key:18s} = {metrics[key]:.4f}')

=== Personalized PageRank — test set ===
  rmse               = 1.0300
  mae                = 0.7781
  hr_at_k            = 0.6508
  ndcg_at_k          = 0.4065
  precision_at_k     = 0.0461
  recall_at_k        = 0.4607
  f1_at_k            = 0.0838


In [8]:
out_path = PREDICTIONS_DIR / 'graph.csv'
pointwise[['userId', 'movieId', 'predicted_rating']].to_csv(out_path, index=False)
print(f'Saved {len(pointwise)} predictions -> {out_path}')

Saved 610 predictions -> /content/gdrive/MyDrive/recsys_artifacts/predictions/graph.csv
